In [23]:
def main():
    parser = argparse.ArgumentParser(description="BenignIDS unified pipeline runner", add_help=True, conflict_handler="resolve")
    parser.add_argument("--stage", type=str, default="all",
        choices=[
            "1.0","2.0","3.0","4.0","5.0","6.0","7.0","8.0","all",  # new
            "0.4","2.1","4.1","4.2","4.3","7.2","8.0-old","9.6"    # aliases
        ],
        help="Pipeline stage to run")
    parser.add_argument("--csv", type=str, default=None, help="CSV path for §1.0 (old 0.4)")
    parser.add_argument("--target", type=str, default=TARGET_COL, help="Target column name")
    parser.add_argument("--f", type=str, default=None, help=argparse.SUPPRESS)  # jupyter noise
    args, _ = parser.parse_known_args()

    # alias map → new section names
    alias = {"0.4":"1.0","2.1":"2.0","4.1":"3.0","4.2":"4.0","4.3":"5.0","7.2":"6.0","8.0-old":"7.0","9.6":"8.0"}
    stage = alias.get(args.stage, args.stage)

    def _auto_discover_csv() -> Optional[Path]:
        try:
            # pick the first readable .csv in CWD
            for p in sorted(Path(".").glob("*.csv")):
                if p.is_file() and os.access(p, os.R_OK):
                    return p.resolve()
        except Exception:
            pass
        return None

    with _timer("Pipeline"):
        # “Top-to-bottom if no parameter” is already the default (`all`);
        # now add auto CSV discovery when splits are not present.
        if stage == "all":
            splits_present = all([
                (SPLITS_DIR/"manifest.json").exists(),
                (SPLITS_DIR/"X_train.parquet").exists(),
                (SPLITS_DIR/"X_val.parquet").exists(),
                (SPLITS_DIR/"y_train.parquet").exists(),
                (SPLITS_DIR/"y_val.parquet").exists(),
            ])
            if not splits_present:
                # Need CSV to start fresh
                csv_path = Path(args.csv) if args.csv else _auto_discover_csv()
                _require(csv_path is not None and csv_path.exists(),
                         "No persisted splits and no CSV found. "
                         "Place a CSV in the current directory or pass --csv.")
                _require(pd is not None, "pandas required")
                df = pd.read_csv(str(csv_path))
                _require(args.target in df.columns, f"Target '{args.target}' not in CSV")
                print(f">>> full pipeline (fresh) using CSV: {csv_path.name}")
                section_1_0_split(df, args.target)
            else:
                print(">>> full pipeline (resume: existing splits detected)")

            section_2_0_payload_hist()
            section_3_0_numeric_matrix()
            feat = _hydrate_features()
            section_4_0_baseline(feat)
            section_5_0_bo(feat)
            section_6_0_cnn(feat)
            section_7_0_reporting()
            section_8_0_champion()
            print(">>> pipeline complete")

        else:
            # Individual stages
            if stage == "1.0":
                if args.csv is None:
                    csv_path = _auto_discover_csv()
                    _require(csv_path is not None, "§1.0 needs --csv or a CSV in the current directory")
                    args.csv = str(csv_path)
                _require(pd is not None, "pandas required")
                df = pd.read_csv(args.csv); _require(args.target in df.columns, f"Target '{args.target}' not in CSV")
                section_1_0_split(df, args.target)
            elif stage == "2.0":
                section_2_0_payload_hist()
            elif stage == "3.0":
                section_3_0_numeric_matrix()
            elif stage in {"4.0","5.0","6.0"}:
                feat = _hydrate_features()
                if stage == "4.0": section_4_0_baseline(feat)
                elif stage == "5.0": section_5_0_bo(feat)
                elif stage == "6.0": section_6_0_cnn(feat)
            elif stage == "7.0":
                section_7_0_reporting()
            elif stage == "8.0":
                section_8_0_champion()
            else:
                raise ValueError(f"Unknown stage: {stage}")
